# Fine-tune the Text-to-SQL model (Step 6b-b)

Trains `Qwen2.5-Coder-1.5B-Instruct` on the pairs in `finetune/data/` with QLoRA, answers the test split with
the base and the fine-tuned model, and exports a GGUF for Ollama.

**Before you start:** Runtime > Change runtime type > **T4 GPU**. Then run the cells in order, one at a time.

Everything is saved to `MyDrive/ida-finetune`. If Colab disconnects, run all the cells again: training continues
from the last finished epoch.

In [ ]:
!nvidia-smi -L

## 1. Install

In [ ]:
!pip install -q unsloth

## 2. Connect Google Drive and upload the data

Allow the Drive access when asked. On the first run, the upload button appears: select `train.jsonl` and
`test.jsonl` from `finetune/data/` on your computer.

In [ ]:
import json
from pathlib import Path

from google.colab import drive, files

drive.mount("/content/drive")
OUT = Path("/content/drive/MyDrive/ida-finetune")
OUT.mkdir(exist_ok=True)

if not all((OUT / name).exists() for name in ("train.jsonl", "test.jsonl")):
    for name, data in files.upload().items():
        (OUT / name).write_bytes(data)

def read_jsonl(path):
    return [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]

train_rows, test_rows = read_jsonl(OUT / "train.jsonl"), read_jsonl(OUT / "test.jsonl")
print(len(train_rows), "train,", len(test_rows), "test")
assert len(train_rows) > 1000 and len(test_rows) > 100

## 3. Settings

`MAX_SEQ_LENGTH` is above the longest example (1,164 tokens) and `MAX_NEW_TOKENS` is above the longest SQL
(257 tokens), so nothing gets cut off. `SYSTEM` must stay the same in `finetune/Modelfile`, because the model
learns with it.

In [ ]:
BASE_MODEL = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
SYSTEM = "You write SQLite queries."
MAX_SEQ_LENGTH = 1280
MAX_NEW_TOKENS = 300
EPOCHS = 3
BATCH_SIZE = 4
LEARNING_RATE = 2e-4
SEED = 3407

## 4. Load the model in 4-bit and add the LoRA layers

QLoRA: the base model stays frozen and compressed to 4 bits, and only small added layers are trained. That is
why a free GPU is enough.

In [ ]:
from unsloth import FastLanguageModel, is_bf16_supported

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL, max_seq_length=MAX_SEQ_LENGTH, dtype=None, load_in_4bit=True)
model = FastLanguageModel.get_peft_model(
    model, r=16, lora_alpha=16, lora_dropout=0, bias="none", random_state=SEED,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth")
model.print_trainable_parameters()

## 5. Turn the pairs into chat text

5% of the training data is held back to watch overfitting during training. The test split is not used here.

In [ ]:
from datasets import Dataset

def chat(prompt, sql=None):
    messages = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": prompt}]
    if sql is not None:
        messages.append({"role": "assistant", "content": sql})
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=sql is None)

data = Dataset.from_list([{"text": chat(r["prompt"], r["completion"])} for r in train_rows])
data = data.train_test_split(test_size=0.05, seed=SEED)
longest = max(len(tokenizer(t).input_ids) for t in data["train"]["text"])
print(len(data["train"]), "to train,", len(data["test"]), "to validate, longest example", longest, "tokens")
assert longest <= MAX_SEQ_LENGTH
print(data["train"][0]["text"][-300:])

## 6. Train

Takes roughly 20-40 minutes on a T4. Watch `eval_loss` after each epoch: if it rises while the training loss
keeps falling, the model is memorising. The best epoch is kept automatically either way.

In [ ]:
from trl import SFTConfig, SFTTrainer

trainer = SFTTrainer(
    model=model, processing_class=tokenizer, train_dataset=data["train"], eval_dataset=data["test"],
    args=SFTConfig(
        output_dir=str(OUT / "checkpoints"), dataset_text_field="text",
        per_device_train_batch_size=BATCH_SIZE, per_device_eval_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=2, num_train_epochs=EPOCHS, learning_rate=LEARNING_RATE, warmup_steps=20,
        optim="adamw_8bit", fp16=not is_bf16_supported(), bf16=is_bf16_supported(), seed=SEED,
        logging_steps=20, eval_strategy="epoch", save_strategy="epoch", save_total_limit=2,
        load_best_model_at_end=True, metric_for_best_model="eval_loss", greater_is_better=False,
        report_to="none"))

resume = any((OUT / "checkpoints").glob("checkpoint-*"))
result = trainer.train(resume_from_checkpoint=resume)
print(result.metrics)
print([(round(log["epoch"]), round(log["eval_loss"], 4)) for log in trainer.state.log_history if "eval_loss" in log])
model.save_pretrained(OUT / "adapter")
tokenizer.save_pretrained(OUT / "adapter")

## 7. Answer the test split with both models

`disable_adapter()` switches the LoRA layers off, which gives the base model without loading it again. The
answers are scored on your computer, where the database is. Takes roughly 5-10 minutes.

In [ ]:
import torch
from tqdm.auto import tqdm

FastLanguageModel.for_inference(model)

def generate(prompt):
    inputs = tokenizer(chat(prompt), return_tensors="pt").to("cuda")
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                                pad_token_id=tokenizer.pad_token_id)
    return tokenizer.decode(output[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

predictions = []
for r in tqdm(test_rows):
    finetuned = generate(r["prompt"])
    with model.disable_adapter():
        base = generate(r["prompt"])
    predictions.append({"question": r["question"], "category": r["category"], "gold": r["completion"],
                        "base": base, "finetuned": finetuned})

path = OUT / "predictions.jsonl"
path.write_text("".join(json.dumps(p, ensure_ascii=False) + "\n" for p in predictions), encoding="utf-8")
for p in predictions[:3]:
    print(f"\nQ: {p['question']}\nGOLD: {p['gold']}\nFINE-TUNED: {p['finetuned']}\nBASE: {p['base']}")

## 8. Export the GGUF

Takes roughly 10-20 minutes and produces a file of about 1 GB.

In [ ]:
model.save_pretrained_gguf(str(OUT / "gguf"), tokenizer, quantization_method="q4_k_m")
for f in (OUT / "gguf").glob("*.gguf"):
    print(f.name, round(f.stat().st_size / 1e9, 2), "GB")

## 9. Download

From `MyDrive/ida-finetune` in Google Drive, download these into `finetune/model/` in the project:

- `predictions.jsonl`
- the `.gguf` file from the `gguf` folder

Both stay out of git: the model is private, and it is too large anyway.